In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import dagshub
import mlflow

pd.set_option('display.max_columns', None)
pd.options.display.max_info_columns = 200
pd.set_option('display.float_format', '{:.4f}'.format)

/home/keras/wordspaces/credit-risk-model/.venv/lib/python3.11/site-packages/mlflow/utils/autologging_utils/versioning.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


### Estas son las varaibles que se determinó eliminar en el EDA

In [7]:
eliminar_vars = [
    'pymnt_plan',              # Soplón: Se crea solo si hay problemas
    'last_fico_range_high',    # Futuro: FICO actualizado (no el de la solicitud)
    'last_fico_range_low',     # Futuro: FICO actualizado (no el de la solicitud)
    'last_credit_pull_d',      # Futuro: Fecha de la última revisión (se actualiza)
    'issue_d',                 # Futuro: Ocurre DESPUÉS de la decisión de aprobar
    'addr_state',              # Variable eliminada por dimensionalidad y poco valor predictivo
    'grade'                    # Variable eliminada por redundancia con sub_grade
]

columnas_a_eliminar_redundancia = [
    'funded_amnt', 
    'funded_amnt_inv', 
    'installment', 
    'fico_range_high',
    'num_sats',
    'num_rev_tl_bal_gt_0'
]

borrar_estas_columnas = eliminar_vars + columnas_a_eliminar_redundancia

# FEATURE ENGINEERING

In [8]:
df = pd.read_parquet("../data/interim/cleaned_loans.parquet")

df.head()

,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,addr_state,dti,delinq_2yrs,earliest_cr_line,fico_range_low,fico_range_high,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,last_credit_pull_d,last_fico_range_high,last_fico_range_low,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,mths_since_rcnt_il,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_rev_tl_bal_gt_0,num_sats,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method
0,3600.0000,3600.0000,3600.0000,36 months,13.9900,123.0300,C,C4,10+ years,MORTGAGE,55000.0000,Not Verified,2015-12-01,0,n,debt_consolidation,PA,5.9100,0.0000,2003-08-01,675.0000,679.0000,1.0000,30.0000,NaN,7.0000,0.0000,2765.0000,29.7000,13.0000,w,2019-03-01,564.0000,560.0000,0.0000,30.0000,Individual,0.0000,722.0000,144904.0000,21.0000,9300.0000,4.0000,20701.0000,1506.0000,37.2000,0.0000,0.0000,148.0000,128.0000,3.0000,3.0000,1.0000,4.0000,69.0000,4.0000,69.0000,2.0000,2.0000,4.0000,2.0000,5.0000,3.0000,4.0000,9.0000,4.0000,7.0000,0.0000,0.0000,0.0000,3.0000,76.9000,0.0000,0.0000,0.0000,178050.0000,7746.0000,2400.0000,13734.0000,Cash
1,24700.0000,24700.0000,24700.0000,36 months,11.9900,820.2800,C,C1,10+ years,MORTGAGE,65000.0000,Not Verified,2015-12-01,0,n,small_business,SD,16.0600,1.0000,1999-12-01,715.0000,719.0000,4.0000,6.0000,NaN,22.0000,0.0000,21470.0000,19.2000,38.0000,w,2019-03-01,699.0000,695.0000,0.0000,NaN,Individual,0.0000,0.0000,204396.0000,19.0000,111800.0000,4.0000,9733.0000,57830.0000,27.1000,0.0000,0.0000,113.0000,192.0000,2.0000,2.0000,4.0000,2.0000,NaN,0.0000,6.0000,0.0000,5.0000,5.0000,13.0000,17.0000,6.0000,20.0000,27.0000,5.0000,22.0000,0.0000,0.0000,0.0000,2.0000,97.4000,7.7000,0.0000,0.0000,314017.0000,39475.0000,79300.0000,24667.0000,Cash
2,20000.0000,20000.0000,20000.0000,60 months,10.7800,432.6600,B,B4,10+ years,MORTGAGE,63000.0000,Not Verified,2015-12-01,0,n,home_improvement,IL,10.7800,0.0000,2000-08-01,695.0000,699.0000,0.0000,NaN,NaN,6.0000,0.0000,7869.0000,56.2000,18.0000,w,2019-03-01,704.0000,700.0000,0.0000,NaN,Joint App,0.0000,0.0000,189699.0000,19.0000,14000.0000,6.0000,31617.0000,2737.0000,55.9000,0.0000,0.0000,125.0000,184.0000,14.0000,14.0000,5.0000,101.0000,NaN,10.0000,NaN,0.0000,2.0000,3.0000,2.0000,4.0000,6.0000,4.0000,7.0000,3.0000,6.0000,0.0000,0.0000,0.0000,0.0000,100.0000,50.0000,0.0000,0.0000,218418.0000,18696.0000,6200.0000,14877.0000,Cash
3,10400.0000,10400.0000,10400.0000,60 months,22.4500,289.9100,F,F1,3 years,MORTGAGE,104433.0000,Source Verified,2015-12-01,0,n,major_purchase,PA,25.3700,1.0000,1998-06-01,695.0000,699.0000,3.0000,12.0000,NaN,12.0000,0.0000,21929.0000,64.5000,35.0000,w,2018-03-01,704.0000,700.0000,0.0000,NaN,Individual,0.0000,0.0000,331730.0000,14.0000,34000.0000,10.0000,27644.0000,4567.0000,77.5000,0.0000,0.0000,128.0000,210.0000,4.0000,4.0000,6.0000,4.0000,12.0000,1.0000,12.0000,0.0000,4.0000,6.0000,5.0000,9.0000,10.0000,7.0000,19.0000,6.0000,12.0000,0.0000,0.0000,0.0000,4.0000,96.6000,60.0000,0.0000,0.0000,439570.0000,95768.0000,20300.0000,88097.0000,Cash
4,11950.0000,11950.0000,11950.0000,36 months,13.4400,405.1800,C,C3,4 years,RENT,34000.0000,Source Verified,2015-12-01,0,n,debt_consolidation,GA,10.2000,0.0000,1987-10-01,690.0000,694.0000,0.0000,

In [9]:
dictionary = pd.read_csv("../data/raw/Lending_Club_Diccionario_Profesional.csv")
dictionary = dict(zip(dictionary['Variable'].str.strip(), dictionary['Descripcion_ES']))

In [10]:
(df.isnull().sum().sort_values(ascending=False)* 100 / len(df)).head(15)

mths_since_last_record           82.9543
mths_since_recent_bc_dlq         76.3096
mths_since_last_major_derog      73.6920
mths_since_recent_revol_delinq   66.5888
mths_since_rcnt_il               60.3850
mths_since_last_delinq           50.3901
mths_since_recent_inq            13.0387
num_tl_120dpd_2m                  8.8722
mo_sin_old_il_acct                7.9667
emp_length                        5.8705
pct_tl_nvr_dlq                    5.1425
avg_cur_bal                       5.1329
mo_sin_rcnt_rev_tl_op             5.1313
mo_sin_old_rev_tl_op              5.1313
num_rev_accts                     5.1313
dtype: float64

En el notebook 01_data_ingestion_and_cleaning, nos dimos cuesta que para estas variables, el NaN significa que "NUNCA" sucedió.

Ejemplo: 'Cuantos meses han pasado desde tal suceso?'
Respuestas: 0, 1, 2, 3, ..., NaN (NUNCA)

Entonces para estas variables de mths_data, rellenaremos con 999 y creamos nueva columna: Si pasó, No pasó

In [11]:
mths_data = [
    'mths_since_last_record',
    'mths_since_recent_bc_dlq',
    'mths_since_last_major_derog',
    'mths_since_recent_revol_delinq',
    'mths_since_rcnt_il',
    'mths_since_last_delinq',
    "mths_since_recent_inq"]

In [12]:
df[mths_data].describe()

,mths_since_last_record,mths_since_recent_bc_dlq,mths_since_last_major_derog,mths_since_recent_revol_delinq,mths_since_rcnt_il,mths_since_last_delinq,mths_since_recent_inq
count,233452.0000,324455.0000,360306.0000,457588.0000,542554.0000,679440.0000,1190992.0000
mean,70.4890,39.6248,43.7008,35.7923,19.6090,34.2822,6.7068
std,26.7754,22.6951,21.3795,22.4233,24.9182,21.9219,5.8484
min,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,53.0000,21.0000,27.0000,17.0000,6.0000,16.0000,2.0000
50%,72.0000,38.0000,44.0000,33.0000,12.0000,31.0000,5.0000
75%,90.0000,58.0000,61.0000,52.0000,22.0000,50.0000,10.0000
max,129.0000,202.0000,226.0000,202.0000,511.0000,226.0000,25.0000


Visualizamos que tenemos 3 variable de tipo Date, Solo nos sirve "earliest_cr_line", puesto que las demás son variables de fuga de datos que se determinó en data cleaning y EDA. 

Queremos aprovechar la informacion.
Sabemos que issue_d contiene fuga de datos, pero se puede usar para calcular la antiguedad del cliente al momento de solicitar un préstamo

In [13]:
date = df.select_dtypes(include=['datetime64[ns]']).columns.tolist()
date

['issue_d', 'earliest_cr_line', 'last_credit_pull_d']

In [30]:
dictionary['earliest_cr_line']

'Fecha (mes/año) en que se abrió la primera línea de crédito del cliente.'

In [29]:
df['earliest_cr_line'].head()

0   2003-08-01
1   1999-12-01
2   2000-08-01
3   1998-06-01
4   1987-10-01
Name: earliest_cr_line, dtype: datetime64[ns]

## CLASE DE FEATURE ENGINEERING

(Primera versión, se va a mejorar)

In [5]:
class FeatureEngineering:
    def __init__(self, dataframe):
        self.df = dataframe.copy()
    def fill_mths_data(self, mths_data_columns):
        for col in mths_data_columns:
            self.df['never_' + col] = self.df[col].isnull().astype(int)
            self.df[col] = self.df[col].fillna(999)
    
    # Calcula la antigüedad crediticia en meses
    def calcular_antiguedad_crediticia(self):
        self.df['earliest_cr_line'] = pd.to_datetime(self.df['earliest_cr_line'])
        self.df['issue_d'] = pd.to_datetime(self.df['issue_d'])
        time_delta = self.df['issue_d'] - self.df['earliest_cr_line']
        self.df["antiguedad_crediticia_meses"] = (time_delta.dt.days / 30.44).fillna(0)
        self.df.drop(columns=['earliest_cr_line'], inplace=True)

    def eliminar_columnas(self, columnas):
        self.df.drop(columns=columnas, inplace=True)

    def run(self, mths_data_columns, columnas_a_eliminar):
        self.fill_mths_data(mths_data_columns)
        self.calcular_antiguedad_crediticia()
        self.eliminar_columnas(columnas_a_eliminar)
        return self.df

In [14]:
fe = FeatureEngineering(df)
data = fe.run(mths_data, borrar_estas_columnas)
data.head()

,loan_amnt,term,int_rate,sub_grade,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,dti,delinq_2yrs,fico_range_low,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,collections_12_mths_ex_med,mths_since_last_major_derog,application_type,acc_now_delinq,tot_coll_amt,tot_cur_bal,mths_since_rcnt_il,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,disbursement_method,never_mths_since_last_record,never_mths_since_recent_bc_dlq,never_mths_since_last_major_derog,never_mths_since_recent_revol_delinq,never_mths_since_rcnt_il,never_mths_since_last_delinq,never_mths_since_recent_inq,antiguedad_crediticia_meses
0,3600.0000,36 months,13.9900,C4,10+ years,MORTGAGE,55000.0000,Not Verified,0,debt_consolidation,5.9100,0.0000,675.0000,1.0000,30.0000,999.0000,7.0000,0.0000,2765.0000,29.7000,13.0000,w,0.0000,30.0000,Individual,0.0000,722.0000,144904.0000,21.0000,9300.0000,4.0000,20701.0000,1506.0000,37.2000,0.0000,0.0000,148.0000,128.0000,3.0000,3.0000,1.0000,4.0000,69.0000,4.0000,69.0000,2.0000,2.0000,4.0000,2.0000,5.0000,3.0000,4.0000,9.0000,0.0000,0.0000,0.0000,3.0000,76.9000,0.0000,0.0000,0.0000,178050.0000,7746.0000,2400.0000,13734.0000,Cash,1,0,0,0,0,0,0,147.9961
1,24700.0000,36 months,11.9900,C1,10+ years,MORTGAGE,65000.0000,Not Verified,0,small_business,16.0600,1.0000,715.0000,4.0000,6.0000,999.0000,22.0000,0.0000,21470.0000,19.2000,38.0000,w,0.0000,999.0000,Individual,0.0000,0.0000,204396.0000,19.0000,111800.0000,4.0000,9733.0000,57830.0000,27.1000,0.0000,0.0000,113.0000,192.0000,2.0000,2.0000,4.0000,2.0000,999.0000,0.0000,6.0000,0.0000,5.0000,5.0000,13.0000,17.0000,6.0000,20.0000,27.0000,0.0000,0.0000,0.0000,2.0000,97.4000,7.7000,0.0000,0.0000,314017.0000,39475.0000,79300.0000,24667.0000,Cash,1,1,1,0,0,0,0,191.9842
2,20000.0000,60 months,10.7800,B4,10+ years,MORTGAGE,63000.0000,Not Verified,0,home_improvement,10.7800,0.0000,695.0000,0.0000,999.0000,999.0000,6.0000,0.0000,7869.0000,56.2000,18.0000,w,0.0000,999.0000,Joint App,0.0000,0.0000,189699.0000,19.0000,14000.0000,6.0000,31617.0000,2737.0000,55.9000,0.0000,0.0000,125.0000,184.0000,14.0000,14.0000,5.0000,101.0000,999.0000,10.0000,999.0000,0.0000,2.0000,3.0000,2.0000,4.0000,6.0000,4.0000,7.0000,0.0000,0.0000,0.0000,0.0000,100.0000,50.0000,0.0000,0.0000,218418.0000,18696.0000,6200.0000,14877.0000,Cash,1,1,1,1,0,1,0,183.9685
3,10400.0000,60 months,22.4500,F1,3 years,MORTGAGE,104433.0000,Source Verified,0,major_purchase,25.3700,1.0000,695.0000,3.0000,12.0000,999.0000,12.0000,0.0000,21929.0000,64.5000,35.0000,w,0.0000,999.0000,Individual,0.0000,0.0000,331730.0000,14.0000,34000.0000,10.0000,27644.0000,4567.0000,77.5000,0.0000,0.0000,128.0000,210.0000,4.0000,4.0000,6.0000,4.0000,12.0000,1.0000,12.0000,0.0000,4.0000,6.0000,5.0000,9.0000,10.0000,7.0000,19.0000,0.0000,0.0000,0.0000,4.0000,96.6000,60.0000,0.0000,0.0000,439570.0000,95768.0000,20300.0000,88097.0000,Cash,1,0,1,0,0,0,0,209.9869
4,11950.0000,36 months,13.4400,C3,4 years,RENT,34000.0000,Source Verified,0,debt_consolidation,10.2000,0.0000,690.0000,0.0000,999.0000,999.0000,5.0000,0.0000,8822.0000,68.4000,6.0000,w,0.0000,999.0000,Individual,0.0000,0.0000,12798.0000,338.0000,12900.0000,0.0000,2560.0000,844.0000,91.0000,0.0000,0.0000,338.0000,54.0000,32.0000,32.0000,0.0000,36.0000,999.0000,999.0000,999.0000,0.0000,2.0000,3.0000,2.0000,2.0000,2.0000,4.0000,4.0000,0.0000,0.0000,0.0

In [ ]:
data.shape

(1369566, 74)

In [27]:
(data.isnull().sum().sort_values(ascending=False)* 100 / len(data)).head(20)

num_tl_120dpd_2m             8.8722
mo_sin_old_il_acct           7.9667
emp_length                   5.8705
pct_tl_nvr_dlq               5.1425
avg_cur_bal                  5.1329
mo_sin_rcnt_rev_tl_op        5.1313
num_rev_accts                5.1313
mo_sin_old_rev_tl_op         5.1313
tot_cur_bal                  5.1313
num_actv_bc_tl               5.1313
num_tl_op_past_12m           5.1313
num_tl_90g_dpd_24m           5.1313
tot_hi_cred_lim              5.1313
total_il_high_credit_limit   5.1313
num_tl_30dpd                 5.1313
num_il_tl                    5.1313
tot_coll_amt                 5.1313
mo_sin_rcnt_tl               5.1313
num_accts_ever_120_pd        5.1313
num_actv_rev_tl              5.1313
dtype: float64

In [15]:
data.describe()

,loan_amnt,int_rate,annual_inc,loan_status,dti,delinq_2yrs,fico_range_low,inq_last_6mths,mths_since_last_delinq,mths_since_last_record,open_acc,pub_rec,revol_bal,revol_util,total_acc,collections_12_mths_ex_med,mths_since_last_major_derog,acc_now_delinq,tot_coll_amt,tot_cur_bal,mths_since_rcnt_il,total_rev_hi_lim,acc_open_past_24mths,avg_cur_bal,bc_open_to_buy,bc_util,chargeoff_within_12_mths,delinq_amnt,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,mort_acc,mths_since_recent_bc,mths_since_recent_bc_dlq,mths_since_recent_inq,mths_since_recent_revol_delinq,num_accts_ever_120_pd,num_actv_bc_tl,num_actv_rev_tl,num_bc_sats,num_bc_tl,num_il_tl,num_op_rev_tl,num_rev_accts,num_tl_120dpd_2m,num_tl_30dpd,num_tl_90g_dpd_24m,num_tl_op_past_12m,pct_tl_nvr_dlq,percent_bc_gt_75,pub_rec_bankruptcies,tax_liens,tot_hi_cred_lim,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,never_mths_since_last_record,never_mths_since_recent_bc_dlq,never_mths_since_last_major_derog,never_mths_since_recent_revol_delinq,never_mths_since_rcnt_il,never_mths_since_last_delinq,never_mths_since_recent_inq,antiguedad_crediticia_meses
count,1369566.0000,1369566.0000,1369139.0000,1369566.0000,1368949.0000,1369537.0000,1369566.0000,1369536.0000,1369566.0000,1369566.0000,1369537.0000,1369537.0000,1369566.0000,1368622.0000,1369537.0000,1369421.0000,1369566.0000,1369536.0000,1299290.0000,1299290.0000,1369566.0000,1299290.0000,1319536.0000,1299267.0000,1305362.0000,1304505.0000,1369421.0000,1369537.0000,1260457.0000,1299289.0000,1299289.0000,1299290.0000,1319536.0000,1306300.0000,1369566.0000,1369566.0000,1369566.0000,1299290.0000,1299290.0000,1299290.0000,1310976.0000,1299290.0000,1299290.0000,1299290.0000,1299289.0000,1248056.0000,1299290.0000,1299290.0000,1299290.0000,1299136.0000,1304949.0000,1368201.0000,1369461.0000,1299290.0000,1319536.0000,1319536.0000,1299290.0000,1369566.0000,1369566.0000,1369566.0000,1369566.0000,1369566.0000,1369566.0000,1369566.0000,1369566.0000
mean,14448.7743,13.2791,76273.6428,0.2123,18.2298,0.3183,696.1020,0.6617,520.4047,840.7289,11.5913,0.2155,16250.9401,51.7738,24.9473,0.0172,747.6795,0.0050,249.1171,140928.4022,611.0139,32763.1818,4.6994,13465.5626,10182.3141,59.8842,0.0091,14.8967,125.7288,181.2912,13.1070,7.8493,1.6636,23.7832,771.7206,136.0892,677.1810,0.5107,3.6442,5.6457,4.7358,8.0869,8.5639,8.2793,14.5896,0.0008,0.0034,0.0890,2.1818,94.1483,45.0947,0.1347,0.0522,174157.8285,49695.3387,21616.5650,42183.7969,0.8295,0.7631,0.7369,0.6659,0.6038,0.5039,0.1304,194.9153
std,8737.5066,4.7841,70308.8730,0.4090,8.9272,0.8793,31.8142,0.9589,482.5915,349.3265,5.4823,0.6040,22438.2667,24.5215,12.0090,0.1473,420.7668,0.0760,10975.7511,157452.7449,479.2734,36549.3674,3.1920,16272.3260,15340.1710,28.2900,0.1097,810.2845,52.3156,94.6095,16.3447,8.7265,1.9964,30.7268,408.0593,334.1786,454.5099,1.3283,2.2516,3.3074,2.9564,4.7980,7.3897,4.5644,8.1192,0.0303,0.0623,0.5050,1.8475,8.7674,36.0144,0.3781,0.4013,177925.5857,47788.0350,21548.1322,43301.0983,0.3760,0.4252,0.4403,0.4717,0.4891,0.5000,0.3367,90.1624
min,500.0000,5.3100,1000.0000,0.0000,0.0000,0.0000,610.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,2.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
25%,8000.0000,9.7500,45760.0000,0.0000,11.7900,0.0000,670.0000,0.0000,31.0000,999.0000,8.0000,0.0000,5927.0000,33.4000,16.0000,0.0000,77.0000,0.0000,0.0000,29367.2500,16.0000,14000.0000,2.0000,3095.0000,1469.0000,38.2000,0.0000,0.0000,98.0000,117.0000,4.0000,3.0000,0.0000,6.0000,999.0000,2.0000,52.0000,0.0000,2.0000,3.0000,3.0000,5.0000,4.0000,5.0000,9.0000,0.0000,0.0000,0.0000,1.0000,91.3000,10.0000,0.0000,0.0000,49567.0000,20881.0000,7800.0000,1475

## GUARDAR DATOS PROCESADOS

In [28]:
# Guardas el nuevo DataFrame intermedio
output_path = "../data/processed/03_feature_engineering.parquet"
data.to_parquet(output_path, index=False)

print(f"DataFrame despues de feature engineering guardado en {output_path}")

DataFrame despues de feature engineering guardado en ../data/processed/03_feature_engineering.parquet
